# 04. Pandas 이해 및 활용 ② DataFrame 변형하기

**AICE Associate 학습 4단계**

읽어온 데이터를 모델이 학습할 수 있는 형태로 바꾸는 작업이 전처리의 핵심입니다.
이번 노트북에서는 컬럼을 **추가·삭제·변경**하고, 값을 **치환**하고, **그룹별로 집계**하는 방법을 익힙니다.

### 학습 목표
1. 컬럼 추가/삭제/이름 변경, 자료형 변환을 할 수 있다.
2. `replace`, `map`, `apply`, `np.where`, `loc`으로 값을 바꿀 수 있다.
3. `groupby`, `pivot_table`, `crosstab`으로 집계할 수 있다.
4. `cut`/`qcut`으로 구간을 나누고, 인덱스를 다룰 수 있다.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('cust_data.csv')
df.head()

,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination
0,B,F,61,2,N,N,7190.9881,3479.1102,2878.7129,N
1,E,M,55,1,N,Y,25045.0340,23940.8983,648.1402,N
2,D,M,29,4,N,Y,24942.9410,20637.5226,4894.4189,N
3,C,M,52,1,Y,N,15789.8007,15655.5138,0.0000,Y
4,C,M,22,2,N,N,9859.3532,7796.0435,1054.4661,N


---
## 1. 컬럼 추가·수정

In [2]:
# 새 컬럼 = 기존 컬럼 연산
df['bill_ratio'] = df['A_bill'] / df['avg_bill']
df['total_AB'] = df['A_bill'] + df['B_bill']
df['age_10s'] = (df['age'] // 10) * 10          # 연령대
df[['age', 'age_10s', 'bill_ratio']].head()

,age,age_10s,bill_ratio
0,61,60,0.483815
1,55,50,0.955914
2,29,20,0.827389
3,52,50,0.991495
4,22,20,0.790726


In [3]:
# 상수값 컬럼 / 위치 지정 삽입
df['year'] = 2026
df.insert(0, 'cust_id', range(1, len(df) + 1))
df.head(3)

,cust_id,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination,bill_ratio,total_AB,age_10s,year
0,1,B,F,61,2,N,N,7190.9881,3479.1102,2878.7129,N,0.483815,6357.8231,60,2026
1,2,E,M,55,1,N,Y,25045.0340,23940.8983,648.1402,N,0.955914,24589.0385,50,2026
2,3,D,M,29,4,N,Y,24942.9410,20637.5226,4894.4189,N,0.827389,25531.9415,20,2026


---
## 2. 컬럼·행 삭제: `drop`

- `axis=1` (또는 `columns=`) → **열** 삭제
- `axis=0` (또는 `index=`) → **행** 삭제 (기본값)
- `inplace=True` → 원본을 직접 변경 (반환값 없음)

> ⚠️ 샘플 문항 7번에서 `drop([...], axis=0)` 을 `axis=1` 로 고치는 문제가 나왔습니다. 컬럼 삭제는 **axis=1**!

In [4]:
df = df.drop(['year', 'cust_id'], axis=1)      # 결과를 다시 할당
df.drop(columns=['total_AB'], inplace=True)     # inplace로 원본 변경
df.columns

Index(['class', 'sex', 'age', 'service', 'stop', 'npay', 'avg_bill', 'A_bill',
       'B_bill', 'termination', 'bill_ratio', 'age_10s'],
      dtype='object')

In [5]:
# 행 삭제: 인덱스를 지정
df.drop([0, 1, 2]).head(3)

,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination,bill_ratio,age_10s
3,C,M,52,1,Y,N,15789.8007,15655.5138,0.0000,Y,0.991495,50
4,C,M,22,2,N,N,9859.3532,7796.0435,1054.4661,N,0.790726,20
5,F,F,21,1,Y,N,25856.9365,25587.5630,0.0000,Y,0.989582,20


In [6]:
# 조건에 맞는 행 삭제: 해당 행들의 index를 drop에 전달 (샘플 문항 5번 패턴)
idx = df[df['age'] > 80].index
print("삭제 대상:", len(idx))
df_tmp = df.drop(idx, axis=0)
print(df.shape, '->', df_tmp.shape)

삭제 대상: 33
(9930, 12) -> (9897, 12)


In [7]:
# 같은 결과: 남길 행만 필터링
df_tmp2 = df[df['age'] <= 80]
print(df_tmp2.shape)

(9897, 12)


---
## 3. 컬럼 이름 바꾸기·자료형 변환

In [8]:
df_ren = df.rename(columns={'avg_bill': 'bill_avg', 'termination': 'churn'})
df_ren.columns

Index(['class', 'sex', 'age', 'service', 'stop', 'npay', 'bill_avg', 'A_bill',
       'B_bill', 'churn', 'bill_ratio', 'age_10s'],
      dtype='object')

In [9]:
df['age'] = df['age'].astype(float)
df['service'] = df['service'].astype(str)
print(df.dtypes)

# 원래대로
df['age'] = df['age'].astype(int)
df['service'] = df['service'].astype(int)

class           object
sex             object
age            float64
service         object
stop            object
npay            object
avg_bill       float64
A_bill         float64
B_bill         float64
termination     object
bill_ratio     float64
age_10s          int64
dtype: object


In [10]:
# 문자열 -> 숫자 (변환 불가한 값은 NaN으로)
s = pd.Series(['10', '20', ' ', 'abc', '35.5'])
pd.to_numeric(s, errors='coerce')

0    10.0
1    20.0
2     NaN
3     NaN
4    35.5
dtype: float64

---
## 4. 값 바꾸기

| 방법 | 용도 |
|---|---|
| `replace({기존: 새값})` | 일부 값만 바꾸기 (나머지는 그대로) |
| `map({기존: 새값})` | 모든 값을 매핑 (매핑에 없으면 NaN) |
| `apply(함수)` | 각 값/행에 함수 적용 |
| `np.where(조건, 참값, 거짓값)` | 조건에 따라 두 값 중 하나 |
| `df.loc[조건, '컬럼'] = 값` | 조건에 맞는 행만 바꾸기 |

In [11]:
df['termination_num'] = df['termination'].map({'Y': 1, 'N': 0})
df[['termination', 'termination_num']].head()

,termination,termination_num
0,N,0
1,N,0
2,N,0
3,Y,1
4,N,0


In [12]:
df['sex'] = df['sex'].replace({'M': 'Male', 'F': 'Female'})
df['sex'].value_counts()

sex
Male      5531
Female    4399
Name: count, dtype: int64

In [13]:
# apply + 사용자 함수
def age_group(age):
    if age < 30:
        return '청년'
    elif age < 60:
        return '중년'
    return '장년'

df['age_group'] = df['age'].apply(age_group)
df['age_group'].value_counts()

age_group
중년    7498
장년    1297
청년    1135
Name: count, dtype: int64

In [14]:
# apply + lambda
df['bill_level'] = df['avg_bill'].apply(lambda x: 'high' if x > 30000 else 'low')
df['bill_level'].value_counts()

bill_level
low     8363
high    1567
Name: count, dtype: int64

In [15]:
# np.where
df['risk'] = np.where((df['npay'] == 'Y') | (df['stop'] == 'Y'), 1, 0)
df['risk'].value_counts()

risk
0    7716
1    2214
Name: count, dtype: int64

In [16]:
# loc + 조건으로 일부만 변경 (샘플 문항 7번 패턴: isin과 함께)
df['grade'] = df['class']
df.loc[df['grade'].isin(['A', 'B', 'C']), 'grade'] = 'low'
df.loc[df['grade'].isin(['D', 'E']), 'grade'] = 'mid'
df.loc[df['grade'].isin(['F', 'G', 'H']), 'grade'] = 'high'
df['grade'].value_counts()

grade
low     3920
mid     3777
high    2233
Name: count, dtype: int64

In [17]:
# 여러 컬럼에 apply (axis=1 : 행 단위로 함수 적용)
df['note'] = df.apply(lambda row: f"{row['class']}-{row['sex'][0]}", axis=1)
df['note'].head()

0    B-F
1    E-M
2    D-M
3    C-M
4    C-M
Name: note, dtype: object

---
## 5. 문자열 처리: `.str` 접근자

In [18]:
t = pd.read_csv('titanic.csv')
t['Title'] = t['Name'].str.extract(r',\s*([^\.]+)\.')   # 이름에서 호칭 추출
t['Title'].value_counts().head()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Dr          7
Name: count, dtype: int64

In [19]:
print(t['Name'].str.contains('Mrs').sum())
print(t['Name'].str.len().max())
t['Cabin'].str[0].value_counts()          # 객실 번호의 첫 글자(갑판)

129
82


Cabin
C    59
B    47
D    33
E    32
A    15
F    13
G     4
T     1
Name: count, dtype: int64

---
## 6. 그룹별 집계: `groupby`

`df.groupby('기준컬럼')['대상컬럼'].집계함수()`

In [20]:
df.groupby('class')['avg_bill'].mean()

class
A    15164.934144
B    17179.989554
C    18985.452512
D    20886.258284
E    22292.725186
F    24161.237663
G    25918.412151
H    28281.486706
Name: avg_bill, dtype: float64

In [21]:
df.groupby(['class', 'termination'])['avg_bill'].mean().head(6)

class  termination
A      N              15710.839233
       Y              12594.451913
B      N              17911.241666
       Y              13726.660203
C      N              19476.678826
       Y              16510.028066
Name: avg_bill, dtype: float64

In [22]:
# agg: 여러 집계를 한 번에
df.groupby('termination').agg({'age': ['mean', 'median'], 'avg_bill': ['mean', 'max'], 'class': 'count'})

age             avg_bill             class
                  mean median          mean         max count
termination                                                  
N            45.680138   45.0  21513.753559  80338.1091  8116
Y            42.521499   43.0  18427.040822  55028.1702  1814

In [23]:
# 결과를 DataFrame으로 (as_index=False 또는 reset_index())
df.groupby('class', as_index=False)['termination_num'].mean().rename(columns={'termination_num': 'churn_rate'})

,class,churn_rate
0,A,0.175172
1,B,0.174749
2,C,0.165583
3,D,0.181357
4,E,0.183230
5,F,0.183861
6,G,0.234652
7,H,0.205374


In [24]:
# transform: 그룹 통계를 원래 행 개수에 맞춰 돌려줌 (결측치를 그룹 평균으로 채울 때 사용)
df['class_avg_bill'] = df.groupby('class')['avg_bill'].transform('mean')
df[['class', 'avg_bill', 'class_avg_bill']].head()

,class,avg_bill,class_avg_bill
0,B,7190.9881,17179.989554
1,E,25045.0340,22292.725186
2,D,24942.9410,20886.258284
3,C,15789.8007,18985.452512
4,C,9859.3532,18985.452512


---
## 7. 피벗 테이블과 교차표

In [25]:
pd.pivot_table(df, index='class', columns='termination', values='avg_bill', aggfunc='mean').round(0)

termination,N,Y
class,,
A,15711.0,12594.0
B,17911.0,13727.0
C,19477.0,16510.0
D,21505.0,18094.0
E,22865.0,19743.0
F,24829.0,21199.0
G,26799.0,23047.0
H,28880.0,25966.0


In [26]:
pd.crosstab(df['class'], df['termination'])

termination,N,Y
class,,
A,598,127
B,987,209
C,1668,331
D,1774,393
E,1315,295
F,799,180
G,561,172
H,414,107


In [27]:
pd.crosstab(df['class'], df['termination'], normalize='index').round(3)   # 행 기준 비율

termination,N,Y
class,,
A,0.825,0.175
B,0.825,0.175
C,0.834,0.166
D,0.819,0.181
E,0.817,0.183
F,0.816,0.184
G,0.765,0.235
H,0.795,0.205


---
## 8. 구간 나누기: `cut`, `qcut`

- `cut`: **값의 경계**를 직접 지정
- `qcut`: **개수가 같도록** 분위수로 나눔

In [28]:
df['age_bin'] = pd.cut(df['age'], bins=[0, 29, 39, 49, 59, 100], labels=['20대이하', '30대', '40대', '50대', '60대이상'])
df['age_bin'].value_counts().sort_index()

age_bin
20대이하    1135
30대      2199
40대      3002
50대      2297
60대이상    1297
Name: count, dtype: int64

In [29]:
df['bill_q'] = pd.qcut(df['avg_bill'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df['bill_q'].value_counts()

bill_q
Q1    2483
Q4    2483
Q2    2482
Q3    2482
Name: count, dtype: int64

---
## 9. 인덱스 다루기와 중복 제거

In [30]:
tmp = df[df['age'] > 70]
tmp.head(3)            # 인덱스가 띄엄띄엄

,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination,...,age_10s,termination_num,age_group,bill_level,risk,grade,note,class_avg_bill,age_bin,bill_q
21,D,Female,73,4,N,N,31273.1968,31260.2583,0.0000,N,...,70,0,장년,high,0,mid,D-F,20886.258284,60대이상,Q4
64,G,Male,73,3,N,N,18422.2661,16402.1203,2062.7388,N,...,70,0,장년,low,0,high,G-M,25918.412151,60대이상,Q2
65,H,Male,79,2,N,N,20871.4172,19977.9573,0.0000,N,...,70,0,장년,low,0,high,H-M,28281.486706,60대이상,Q3


In [31]:
tmp = tmp.reset_index(drop=True)    # drop=True : 기존 인덱스를 컬럼으로 남기지 않음
tmp.head(3)

,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination,...,age_10s,termination_num,age_group,bill_level,risk,grade,note,class_avg_bill,age_bin,bill_q
0,D,Female,73,4,N,N,31273.1968,31260.2583,0.0000,N,...,70,0,장년,high,0,mid,D-F,20886.258284,60대이상,Q4
1,G,Male,73,3,N,N,18422.2661,16402.1203,2062.7388,N,...,70,0,장년,low,0,high,G-M,25918.412151,60대이상,Q2
2,H,Male,79,2,N,N,20871.4172,19977.9573,0.0000,N,...,70,0,장년,low,0,high,H-M,28281.486706,60대이상,Q3


In [32]:
df.set_index('class').head(3)

,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination,bill_ratio,age_10s,termination_num,age_group,bill_level,risk,grade,note,class_avg_bill,age_bin,bill_q
class,,,,,,,,,,,,,,,,,,,,
B,Female,61,2,N,N,7190.9881,3479.1102,2878.7129,N,0.483815,60,0,장년,low,0,low,B-F,17179.989554,60대이상,Q1
E,Male,55,1,N,Y,25045.0340,23940.8983,648.1402,N,0.955914,50,0,중년,low,1,mid,E-M,22292.725186,50대,Q3
D,Male,29,4,N,Y,24942.9410,20637.5226,4894.4189,N,0.827389,20,0,청년,low,1,mid,D-M,20886.258284,20대이하,Q3


In [33]:
dup = pd.concat([df.head(3), df.head(2)])
print(dup.duplicated().sum())                 # 중복 행 개수
dup.drop_duplicates().shape

2


(3, 21)

---
## ✏️ 연습문제 (`cust_data.csv`를 새로 읽어서 사용)

In [34]:
data = pd.read_csv('cust_data.csv')

**문제 1.** `termination` 컬럼 값을 Y→1, N→0 으로 바꾸어 같은 컬럼에 저장하고, 1의 개수를 `답안1` 변수에 저장하세요.

In [35]:
# 여기에 답안코드를 작성하고 실행하세요.

**문제 2.** `A_bill`과 `B_bill`의 합이 `avg_bill`에서 차지하는 비율을 `AB_ratio` 컬럼으로 추가하세요.

In [36]:
# 여기에 답안코드를 작성하고 실행하세요.

**문제 3.** 고객 등급(`class`)별 해지율(`termination`의 평균)을 구하고, 해지율이 가장 높은 등급을 `답안3` 변수에 저장하세요.

In [37]:
# 여기에 답안코드를 작성하고 실행하세요.

**문제 4.** 아래 코드의 빈칸을 채워, `stop`, `npay` 컬럼을 삭제하고 index를 초기화한 데이터를 `data2`에 저장하세요.
```python
data2 = data.< blank4-1 >(['stop', 'npay'], axis=< blank4-2 >)
data2.reset_index(drop=True, inplace=True)
```
빈칸에 들어갈 내용을 `답안4_1`, `답안4_2` 변수에 저장하세요.

In [38]:
# 아래 코드의 빈칸을 채우고 실행하세요.

**문제 5.** `pd.cut`을 사용해 `age`를 `[0, 40, 60, 100]` 구간, 라벨 `['young', 'middle', 'senior']`로 나눈 `age_cat` 컬럼을 만들고, 구간별 해지율을 출력하세요.

In [39]:
# 여기에 답안코드를 작성하고 실행하세요.

---
## ✅ 정답 및 해설

In [40]:
# 문제 1
data = pd.read_csv('cust_data.csv')
data['termination'] = data['termination'].map({'Y': 1, 'N': 0})
답안1 = data['termination'].sum()
print(답안1)

# 문제 2
data['AB_ratio'] = (data['A_bill'] + data['B_bill']) / data['avg_bill']
print(data['AB_ratio'].describe())

# 문제 3
churn_by_class = data.groupby('class')['termination'].mean().sort_values(ascending=False)
print(churn_by_class)
답안3 = churn_by_class.index[0]
print('답안3 =', 답안3)

1814
count    9930.000000
mean        1.001571
std         0.046762
min         0.434310
25%         0.979950
50%         0.999430
75%         1.020504
max         1.600543
Name: AB_ratio, dtype: float64
class
G    0.234652
H    0.205374
F    0.183861
E    0.183230
D    0.181357
A    0.175172
B    0.174749
C    0.165583
Name: termination, dtype: float64
답안3 = G


In [41]:
# 문제 4
data2 = data.drop(['stop', 'npay'], axis=1)
data2.reset_index(drop=True, inplace=True)
답안4_1 = 'drop'
답안4_2 = 1
data2.head(3)

,class,sex,age,service,avg_bill,A_bill,B_bill,termination,AB_ratio
0,B,F,61,2,7190.9881,3479.1102,2878.7129,0,0.884138
1,E,M,55,1,25045.0340,23940.8983,648.1402,0,0.981793
2,D,M,29,4,24942.9410,20637.5226,4894.4189,0,1.023614


In [42]:
# 문제 5
data['age_cat'] = pd.cut(data['age'], bins=[0, 40, 60, 100], labels=['young', 'middle', 'senior'])
data.groupby('age_cat', observed=True)['termination'].mean()

age_cat
young     0.223115
middle    0.159325
senior    0.160821
Name: termination, dtype: float64